# 01 — Data Collection

Fetches all raw data for the Austria Energy & Climate project.

| Source | What we get | Credentials needed |
|---|---|---|
| Our World in Data | Annual energy & CO₂ time series | None |
| Open-Meteo / ERA5 | Hourly weather (Vienna) | None |
| Eurostat (env_air_gge) | Annual GHG emissions by sector | None |
| EEA — Effort Sharing (DAT-170-en) | Annual non-ETS emissions & 2030 target | None |
| ENTSO-E | Hourly generation, load, prices | API key (free) |

**Run order:** all sources except ENTSO-E can be ran immediately, no credentials needed. 

**Pre-requisite for ENTSO-E:** Add your API key to `.env`:
```
ENTSOE_API_KEY=your-key-here
```
Then re-run the Setup cell and continue from ENTSO-E section.

## Setup

Install once if needed: `pip install entsoe-py requests pandas python-dotenv eurostat` (or the conda-forge equivalents).

In [ ]:
# imports, project path, environment, and the (keyless) data loader
import sys
from pathlib import Path

import eurostat
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import DataLoader  # noqa: E402

load_dotenv(PROJECT_ROOT / ".env")  # ENTSOE_API_KEY, used in the ENTSO-E section  # pyright: ignore[reportUnusedCallResult]

# no API key needed for OWID / weather / Eurostat / EEA
dl = DataLoader()

# QA sanity-check figures are committed here (data/raw/ is for raw data only)
FIG_QA = PROJECT_ROOT / "figures" / "qa"
FIG_QA.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

## 1 · Our World in Data — annual energy series

Downloads the full OWID energy CSV (~10 MB) and saves the Austria slice.  
No credentials required.

In [ ]:
# fetch the OWID energy CSV, keep the Austria slice
owid_path = dl.fetch_owid()

df_owid = pd.read_csv(owid_path, index_col="year")
print(f"Shape: {df_owid.shape}")
df_owid.tail()

In [ ]:
# Quick sanity check: which columns do we have?
energy_cols = [c for c in df_owid.columns if any(
    kw in c for kw in ["electricity", "renewables", "carbon", "energy", "co2", "solar", "wind", "hydro"]
)]
print(f"{len(energy_cols)} energy-related columns:")
for c in energy_cols:
    print(" ", c)

## 2 · Open-Meteo — hourly weather (Vienna)

ERA5 reanalysis data via the free Open-Meteo API.  
No account or API key needed.

In [ ]:
# fetch hourly ERA5 weather for Vienna (Open-Meteo)
START = "2019-01-01"
END   = "2024-12-31"

weather_path = dl.fetch_weather(start=START, end=END)

df_weather = pd.read_csv(weather_path, index_col="timestamp", parse_dates=True)
print(f"Shape: {df_weather.shape}")
print(f"Date range: {df_weather.index.min()} → {df_weather.index.max()}")
df_weather.head()

In [ ]:
# Basic stats + missing value check
print("=== Missing values ===")
print(df_weather.isna().sum())
print()
print("=== Descriptive stats ===")
df_weather.describe().round(2)

In [ ]:
# daily-mean temperature & solar radiation — confirms the series looks seasonal
fig, (ax_temp, ax_solar) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

daily = df_weather.resample("D").mean()

ax_temp.plot(daily.index, daily["temperature_2m"], lw=0.8, color="#E8593C")
ax_temp.set_ylabel("Temperature (°C)")
ax_temp.set_title("Vienna — daily mean temperature & solar radiation (ERA5)")

ax_solar.plot(daily.index, daily["shortwave_radiation"], lw=0.8, color="#EF9F27")
ax_solar.set_ylabel("Solar radiation (W/m²)")

for ax in (ax_temp, ax_solar):
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_QA / "weather_sanity_check.png", dpi=120)
plt.show()
print("Looks reasonable? Seasonal oscillations present -> ok")

## 3 · Eurostat — GHG emissions inventory

Austria's national greenhouse-gas inventory (dataset `env_air_gge`) — the official
UNFCCC figures, re-published by Eurostat. Annual, 1990 → latest (~2 years behind),
all gases combined as CO₂-equivalent. No API key needed.

Feeds **RQ6** (emissions trajectory vs the 2030 target). We pull *all* sectors and
units here and pick what we need during cleaning, so this cell doubles as a first
look at what Eurostat actually returns.

In [ ]:
# ── Fetch GHG inventory + first look ──────────────────────────────────────────
# Keyless; reuses `dl` from Setup. Pulls all sectors/units for AT,
# GHG aggregate (CO₂-eq), every year.
ghg_path = dl.fetch_ghg("AT")
df_ghg = pd.read_csv(ghg_path)

print(f"Shape:         {df_ghg.shape}")
print(f"Columns:       {list(df_ghg.columns)}")
print(f"Units present: {df_ghg['unit'].unique()}")
print()

# Cryptic sector codes → human labels, to spot the TOTAL during cleaning
sector_dic = eurostat.get_dic("env_air_gge", "src_crf", frmt="dict")  # {code: label}
print("Sectors present in the Austria data:")
for code in df_ghg["src_crf"].unique():
    print(f"  {code:<18} {sector_dic.get(code, '?')}")

df_ghg.head()

## 4 · EEA — Effort Sharing (non-ETS) emissions · for RQ6

Non-ETS greenhouse-gas emissions and the 2030 Effort Sharing target
(EEA dataset DAT-170-en, CC-BY 4.0). Annual, no credentials.
The workbook's structure is inspected and loaded into DuckDB in notebook 02.

In [ ]:
# fetch the EEA Effort Sharing (non-ETS) workbook
esr_path = dl.fetch_esr()

size_kb = esr_path.stat().st_size / 1024
print(f"Saved: {esr_path.name}  ({size_kb:.0f} KB)")

## 5 · ENTSO-E — hourly generation, load & prices

If the key is not set yet, this section will log a warning and skip — that's fine.

In [ ]:
# re-check the loader now that .env is loaded and carries the API key
# re-run the setup cell if needed
if dl._entsoe_client is None:
    print("⚠️  ENTSO-E client not available — add your key to .env and re-run.")
else:
    print("✓  ENTSO-E client ready.")

In [ ]:
# ── Fetch generation ──────────────────────────────────────────────────────────
# Takes ~2–4 minutes for 2019–2024 (6 API calls, one per year).
# Safe to interrupt and resume — each year chunk is independent.
if dl._entsoe_client:
    ts_start = pd.Timestamp(START, tz="UTC")
    ts_end   = pd.Timestamp(END,   tz="UTC") + pd.Timedelta(days=1)

    gen_path = dl.fetch_generation(ts_start, ts_end)
    assert gen_path is not None
    df_gen = pd.read_csv(
        gen_path,
        header=[0, 1],          # 2-level columns: fuel × (Actual Aggregated / Consumption)
        index_col=0,
        parse_dates=True,
    )
    df_gen.index = pd.to_datetime(df_gen.index, utc=True)
    df_gen = df_gen.sort_index()
    print(f"Shape: {df_gen.shape}")
    print("Columns (fuel types):", df_gen.columns.tolist())
    df_gen.head()

In [ ]:
# ── Fetch load ────────────────────────────────────────────────────────────────
if dl._entsoe_client:
    load_path = dl.fetch_load(ts_start, ts_end)
    assert load_path is not None
    df_load = pd.read_csv(load_path, index_col=0, parse_dates=True)
    df_load.index = pd.to_datetime(df_load.index, utc=True)
    df_load = df_load.sort_index()
    print(f"Shape: {df_load.shape}")
    df_load.head()

In [ ]:
# ── Fetch day-ahead prices ────────────────────────────────────────────────────
if dl._entsoe_client:
    prices_path = dl.fetch_prices(ts_start, ts_end)
    assert prices_path is not None
    df_prices = pd.read_csv(prices_path, index_col=0, parse_dates=True)
    # timestamps carry explicit offsets (+01:00/+02:00) → UTC is unambiguous across DST
    df_prices.index = pd.to_datetime(df_prices.index, utc=True)
    df_prices = df_prices.sort_index()
    print(f"Shape: {df_prices.shape}")
    df_prices.head()

In [ ]:
# ── one sample week of the generation mix (sanity check) ──
if dl._entsoe_client:
    # select the "Actual Aggregated" level from the 2-level (fuel × flow) columns
    week = df_gen.xs("Actual Aggregated", axis=1, level=1).loc["2023-07-01":"2023-07-07"]
    week = week.dropna(axis=1, how="all")   # keep only fuels with data

    fig, ax = plt.subplots(figsize=(14, 5))
    week.plot.area(ax=ax, stacked=True, linewidth=0)
    ax.set_title("Austria — hourly generation mix (sample week, July 2023)")
    ax.set_ylabel("MW")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="upper right", fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(FIG_QA / "generation_sanity_check.png", dpi=120)
    plt.show()

In [ ]:
# ── Hydro Pumped Storage: net flow (generation − consumption) ──
# Positive = turbines (sending to grid); negative = pumping (drawing from grid)
if dl._entsoe_client:
    gen  = df_gen[("Hydro Pumped Storage", "Actual Aggregated")]
    cons = df_gen[("Hydro Pumped Storage", "Actual Consumption")]
    net  = (gen - cons).loc["2023-07-01":"2023-07-07"]

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.fill_between(net.index, net.values, 0,  # pyright: ignore[reportArgumentType]
                    where=(net.values >= 0), color="#2ca02c", alpha=0.7,  # pyright: ignore[reportOperatorIssue, reportArgumentType]
                    label="Generating (turbine)")
    ax.fill_between(net.index, net.values, 0,  # pyright: ignore[reportArgumentType]
                    where=(net.values < 0),  color="#d62728", alpha=0.7,  # pyright: ignore[reportOperatorIssue, reportArgumentType]
                    label="Pumping (consuming)")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title("Austria — Hydro Pumped Storage net flow (sample week, July 2023)")
    ax.set_ylabel("MW   (+ generating  /  − pumping)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="upper right", fontsize=9)
    plt.tight_layout()
    plt.show()

## 6 · Summary — what we have

Run this cell after all fetches to confirm everything landed.

In [ ]:
# confirm everything landed
raw_dir = PROJECT_ROOT / "data" / "raw"
ext_dir = PROJECT_ROOT / "data" / "external"

print("=== data/raw/ ===")
for f in sorted(raw_dir.glob("*.csv")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<40}  {size_kb:>8.1f} KB")

print()
print("=== data/external/ ===")
for f in sorted(ext_dir.glob("*.csv")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<40}  {size_kb:>8.1f} KB")

print()
print("Data collection complete ✓")
print("Next: notebook 02_data_cleaning.ipynb")